In [2]:
from attention_predict.reconstruct_mini import reconstruct_traces, build_mini_attention_model, build_dataloader
import json
import matplotlib.pyplot as plt
import numpy as np
import torch

In [3]:
attention_scheme = 'BfromN'
depth = 5
num_neurons = 71
num_behaviors = 8
window_size = 400
num_fmaps = 64
device = 'cuda:1'
ds_type = 'valid'
ds_name = f'data0715e_norm0505_{ds_type}'
perturb_index = None

model_ckpt = 91
tag = 'currbest_fix'
experiment = f'whole_brain/depth{depth}_nfmaps{num_fmaps}_lr5e-05_{attention_scheme}-8B_norm0505_{tag}'

if ds_type == 'valid':
    num_worms = 27 # could be 16
elif ds_type == 'test':
    num_worms = 20 # could be 10
else:
    num_worms = 94 # could be 64

In [4]:
dataloader = build_dataloader(ds_name, device)
model = build_mini_attention_model(
    attention_scheme,
    depth,
    num_neurons,
    num_behaviors,
    window_size,
    num_fmaps,
    None,
    device
)
reconstructed_traces = reconstruct_traces(
    model,
    attention_scheme,
    dataloader,
    model_ckpt,
    experiment,
    num_worms,
    num_neurons,
    num_behaviors,
    device,
    perturb_index=perturb_index,
)

'\nreconstructed_traces = reconstruct_traces(\n    model,\n    attention_scheme,\n    dataloader,\n    model_ckpt,\n    experiment,\n    num_worms,\n    num_neurons,\n    num_behaviors,\n    device,\n    perturb_index=perturb_index,\n)\n'

In [5]:
def plot_reconstruction(
    worm_index,
    reconstructed_traces,
    xrange,
    output_variables,
    context_variables,
    attention_scheme,
    num_neurons,
    num_behaviors,
):
    num_windows = len(reconstructed_traces[worm_index]['prediction'])
    ground_truth_activity = [
        np.zeros_like(reconstructed_traces[worm_index]['prediction'][k])
        for k in range(num_windows)
    ]
    ground_truth_2channels = reconstructed_traces[worm_index]['ground_truth']
    num_vecs = ground_truth_2channels[0].shape[1]

    for k in range(num_windows):
        for i, j in enumerate(range(0, num_vecs, 2)):
            ground_truth_activity[k][:, i, :] = ground_truth_2channels[k][:, j, :][:]
    concatenated_prediction = np.concatenate(
        reconstructed_traces[worm_index]['prediction'],
        axis=2
    )
    concatenated_ground_truth = np.concatenate(
        ground_truth_activity,
        axis=2
    )
    prediction = concatenated_prediction.squeeze(0)
    ground_truth = concatenated_ground_truth.squeeze(0)
    
    masked_outcomes = {}
    masked_outcomes['prediction'] = prediction
    masked_outcomes['ground_truth'] = ground_truth

    num_output_variables = len(output_variables)
    num_context_variables = len(context_variables)
    
    fig, axs = plt.subplots(
        num_output_variables, 1,
        figsize=(20, num_output_variables * 2),
        sharex=True
    )
    for i in range(num_output_variables):
        
        axs[i].plot(ground_truth[i][:xrange], label='ground truth', alpha=0.8)
        axs[i].plot(prediction[i][:xrange], label='prediction')
        axs[i].set_title(
            f'{output_variables[i]} from {datasets[worm_index]}')
        axs[i].grid(True)

    return masked_outcomes

In [6]:
xrange = 1600

fig4_neurons = ['RIB', 'RIC', 'RID', 'AUA', 'AVJ', 'AVK', 'AIM', 'AIY',
                'AVA', 'AVE', 'AIB', 'RIM', 'RIV', 'ADA', 'AVD', 'RIA',
                'AVH', 'AIN', 'AIZ', 'URB', 'ALA', 'RMG', 'RMD', 'RMDD',
                'RMDV', 'RME', 'RMEV', 'RMED', 'SAAV', 'SMDV', 'IL1L',
                'IL1R', 'IL1D', 'IL1V', 'URYD', 'URYV', 'BAG', 'ASG',
                'CEPD', 'CEPV', 'OLL', 'OLQD', 'OLQV', 'IL2L', 'IL2R',
                'IL2D', 'IL2V', 'URAD', 'URAV', 'ADE', 'FLP', 'AQR',
                'URX', 'ADL', 'ASH', 'ASEL', 'ASER', 'AWA', 'AWB',
                'AWC', 'I1', 'I2', 'I3', 'NSM', 'M1', 'M3', 'M4',
                'M5', 'MC', 'MI']

if attention_scheme == 'NfromB':
    output_variables = {
        i: neuron for i, neuron in enumerate(fig4_neurons)
    }
    context_variables = {0: 'heatstim', 1: 'velocity', 2: 'pumping', 3: 'head angle'}

if attention_scheme == 'NfromN':
    output_variables = {
        i: neuron for i, neuron in enumerate(fig4_neurons)
    }
    context_variables = output_variables

if attention_scheme == 'BfromN':
    # other possibilities:
    # output_variables = {0: 'velocity', 1: 'pumping', 2: 'head angle'}
    # output_variables = {0: 'heatstim', 1: 'velocity', 2: 'pumping', 3: 'head angle'}
    output_variables = {
        0: 'velocity', 1: 'pumping', 2: 'head angle',
        3: 'body angle 0', 4: 'body angle 1', 5: 'body angle 2',
        6: 'turning rate', 7: 'worm angle'
    }
    context_variables = {
        i: neuron for i, neuron in enumerate(fig4_neurons)
    }

In [ ]:
worm_index = np.random.choice(list(range(num_worms)))
print(reconstructed_traces[worm_index]['dataset'])
print(f'worm index: {worm_index}')

prj_directory = '/store1/alicia/attention_predict'
data_path = f'{prj_directory}/data/{ds_name}.npy'
dataset_path = f'{prj_directory}/data/{ds_name}_ds.npy'
datasets = np.load(dataset_path)
num_worms = np.load(data_path).shape[0]

In [ ]:
masked_outcomes = plot_reconstruction(
    worm_index,
    reconstructed_traces,
    xrange,
    output_variables,
    context_variables,
    attention_scheme,
    num_neurons,
    num_behaviors,
)